In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

train = pd.read_csv('train.csv', sep = ',')
eval = pd.read_csv('test.csv', sep = ',')

pd.options.plotting.backend = "plotly"
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
train.plot(kind='hist', y='Survived', x='Sex')

In [3]:
X = train.copy().drop(columns=['Survived'])
y = train['Survived']

In [4]:
np.mean((X['Sex'] == 'female').astype(int) == y)

np.float64(0.7867564534231201)

In [5]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Name         891 non-null    object 
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Cabin        204 non-null    object 
 10  Embarked     889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 76.7+ KB


In [6]:
from sklearn.neighbors import KNeighborsClassifier

X_select = X[['Pclass', 'SibSp', 'Parch', 'Fare']]

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_select, y)
np.mean(knn.predict(X_select) == y)

np.float64(0.7687991021324355)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_select, y, test_size=0.2, random_state=42)
knn.fit(X_train, y_train)
np.mean(knn.predict(X_test) == y_test)

np.float64(0.7094972067039106)

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(knn, X_select, y, cv=skf)
cv_scores, cv_scores.mean()

(array([0.66480447, 0.70786517, 0.62921348, 0.61797753, 0.64606742]),
 np.float64(0.6531856129558722))

In [9]:
scores = []

for i, (train_index, test_index) in enumerate(skf.split(X_select, y)):
    X_train, X_test = X_select.iloc[train_index], X_select.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    knn.fit(X_train, y_train)
    score = np.mean(knn.predict(X_test) == y_test)
    scores.append(score)
    print(f"Fold {i} : {score}")

print("Moyenne score :", np.mean(scores))

Fold 0 : 0.664804469273743
Fold 1 : 0.7078651685393258
Fold 2 : 0.6292134831460674
Fold 3 : 0.6179775280898876
Fold 4 : 0.6460674157303371
Moyenne score : 0.6531856129558722


In [10]:
scores = []

for i, (train_index, test_index) in enumerate(skf.split(X_select, y)):
    X_train, X_test = X_select.iloc[train_index], X_select.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    score = np.mean(knn.predict(X_test) == y_test)
    scores.append(score)
    print(f"Fold {i} : {score}")

print("Moyenne score :", np.mean(scores))

Fold 0 : 0.6759776536312849
Fold 1 : 0.7078651685393258
Fold 2 : 0.6235955056179775
Fold 3 : 0.6404494382022472
Fold 4 : 0.6573033707865169
Moyenne score : 0.6610382273554705


In [11]:
from sklearn.linear_model import LogisticRegression

for i, (train_index, test_index) in enumerate(skf.split(X_select, y)):
    X_train, X_test = X_select.iloc[train_index], X_select.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LogisticRegression()
    model.fit(X_train, y_train)
    score = np.mean(model.predict(X_test) == y_test)
    scores.append(score)
    print(f"Fold {i} : {score}")

print("Moyenne score :", np.mean(scores))

Fold 0 : 0.659217877094972
Fold 1 : 0.6966292134831461
Fold 2 : 0.702247191011236
Fold 3 : 0.6910112359550562
Fold 4 : 0.6629213483146067
Moyenne score : 0.6717218002636368


In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_select, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [13]:
np.mean(model.predict(X_test) == y_test)

np.float64(0.7150837988826816)

In [14]:
d = model.predict_proba(X_test)
d

array([[0.72546632, 0.27453368],
       [0.60834045, 0.39165955],
       [0.74978611, 0.25021389],
       [0.51708491, 0.48291509],
       [0.77773812, 0.22226188],
       [0.36794299, 0.63205701],
       [0.74994793, 0.25005207],
       [0.80069255, 0.19930745],
       [0.74994793, 0.25005207],
       [0.30916839, 0.69083161],
       [0.43962244, 0.56037756],
       [0.74967049, 0.25032951],
       [0.77974448, 0.22025552],
       [0.75043297, 0.24956703],
       [0.60540005, 0.39459995],
       [0.35260213, 0.64739787],
       [0.44028542, 0.55971458],
       [0.74987471, 0.25012529],
       [0.60540005, 0.39459995],
       [0.39922767, 0.60077233],
       [0.74981312, 0.25018688],
       [0.42966485, 0.57033515],
       [0.78061151, 0.21938849],
       [0.74858185, 0.25141815],
       [0.74365593, 0.25634407],
       [0.72931087, 0.27068913],
       [0.42825101, 0.57174899],
       [0.60540005, 0.39459995],
       [0.72931087, 0.27068913],
       [0.74985159, 0.25014841],
       [0.

In [15]:
d[:, 0] + d[:, 1]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [16]:
p =d[:, 1]
np.mean((p >= 0.50) == y_test)

np.float64(0.7150837988826816)

In [17]:
import plotly.graph_objects as go

thresholds = np.arange(0, 1.01, 0.005)
accuracies = [np.mean((p >= t) == y_test) for t in thresholds]

fig = go.Figure(data=go.Scatter(x=thresholds, y=accuracies))
fig.show()


In [18]:
from sklearn.metrics import accuracy_score

thresholds = np.arange(0, 1.01, 0.005)
accuracies = [accuracy_score(y_test, p >= t) for t in thresholds]

fig = go.Figure(data=go.Scatter(x=thresholds, y=accuracies))
fig.show()


In [19]:
from sklearn.metrics import f1_score

thresholds = np.arange(0, 1.01, 0.005)
f1s = [f1_score(y_test, p >= t) for t in thresholds]

fig = go.Figure(data=go.Scatter(x=thresholds, y=f1s))
fig.show()


In [20]:
thresholds = np.arange(0, 1.01, 0.005)
f1s = []

for t in thresholds:
    f1s_t = []

    for i, (train_index, test_index) in enumerate(skf.split(X_select, y)):
        X_train, X_test = X_select.iloc[train_index], X_select.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model = LogisticRegression()
        model.fit(X_train, y_train)
        p = model.predict_proba(X_test)[:, 1]

        f1s_t.append(f1_score(y_test, p >= t))

    f1s.append(np.mean(f1s_t))

fig = go.Figure(data=go.Scatter(x=thresholds, y=f1s))
fig.show()

In [21]:
thresholds = np.arange(0, 1.01, 0.005)
accuracies = []

for t in thresholds:
    accuracies_t = []

    for i, (train_index, test_index) in enumerate(skf.split(X_select, y)):
        X_train, X_test = X_select.iloc[train_index], X_select.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model = LogisticRegression()
        model.fit(X_train, y_train)
        p = model.predict_proba(X_test)[:, 1]

        accuracies_t.append(accuracy_score(y_test, p >= t))

    accuracies.append(np.mean(accuracies_t))

fig = go.Figure(data=go.Scatter(x=thresholds, y=accuracies))
fig.show()

In [22]:
thresholds[np.argmax(accuracies)]

np.float64(0.41500000000000004)

In [23]:
from sklearn.linear_model import LogisticRegression

scores = []

for i, (train_index, test_index) in enumerate(skf.split(X_select, y)):
    X_train, X_test = X_select.iloc[train_index], X_select.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LogisticRegression()
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    prediction = np.where(p >= 0.415, 1, 0)
    score = accuracy_score(y_test, prediction)
    scores.append(score)
    print(f"Fold {i} : {score}")

print("Moyenne score :", np.mean(scores))

Fold 0 : 0.6759776536312849
Fold 1 : 0.7247191011235955
Fold 2 : 0.6966292134831461
Fold 3 : 0.7078651685393258
Fold 4 : 0.6910112359550562
Moyenne score : 0.6992404745464815


In [24]:
X.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [25]:
X.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [26]:
X.isna().plot(kind='imshow')

In [27]:
eval.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [29]:
eval.isna().plot(kind='imshow')

In [30]:
X['Age'].fillna(X['Age'].mean())

0      22.000000
1      38.000000
2      26.000000
3      35.000000
4      35.000000
         ...    
886    27.000000
887    19.000000
888    29.699118
889    26.000000
890    32.000000
Name: Age, Length: 891, dtype: float64

In [31]:
X.select_dtypes("number").corr()

,PassengerId,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.000000,-0.035144,0.036847,-0.057527,-0.001652,0.012658
Pclass,-0.035144,1.000000,-0.369226,0.083081,0.018443,-0.549500
Age,0.036847,-0.369226,1.000000,-0.308247,-0.189119,0.096067
SibSp,-0.057527,0.083081,-0.308247,1.000000,0.414838,0.159651
Parch,-0.001652,0.018443,-0.189119,0.414838,1.000000,0.216225
Fare,0.012658,-0.549500,0.096067,0.159651,0.216225,1.000000


In [32]:
X.select_dtypes("number").corr().abs()

,PassengerId,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.000000,0.035144,0.036847,0.057527,0.001652,0.012658
Pclass,0.035144,1.000000,0.369226,0.083081,0.018443,0.549500
Age,0.036847,0.369226,1.000000,0.308247,0.189119,0.096067
SibSp,0.057527,0.083081,0.308247,1.000000,0.414838,0.159651
Parch,0.001652,0.018443,0.189119,0.414838,1.000000,0.216225
Fare,0.012658,0.549500,0.096067,0.159651,0.216225,1.000000


In [33]:
X.select_dtypes("number").corr().abs().plot(kind='imshow')

In [34]:
X['Sex'] = X['Sex'].eq('female').astype(int)
eval['Sex'] = eval['Sex'].eq('female').astype(int)

In [35]:
X.select_dtypes("number").corr().abs().plot(kind='imshow')

In [36]:
X.select_dtypes("number").corr().abs()

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare
PassengerId,1.000000,0.035144,0.042939,0.036847,0.057527,0.001652,0.012658
Pclass,0.035144,1.000000,0.131900,0.369226,0.083081,0.018443,0.549500
Sex,0.042939,0.131900,1.000000,0.093254,0.114631,0.245489,0.182333
Age,0.036847,0.369226,0.093254,1.000000,0.308247,0.189119,0.096067
SibSp,0.057527,0.083081,0.114631,0.308247,1.000000,0.414838,0.159651
Parch,0.001652,0.018443,0.245489,0.189119,0.414838,1.000000,0.216225
Fare,0.012658,0.549500,0.182333,0.096067,0.159651,0.216225,1.000000


In [37]:
X.groupby(['Pclass', 'Sex'])['Age'].mean()

Pclass  Sex
1       0      41.281386
        1      34.611765
2       0      30.740707
        1      28.722973
3       0      26.507589
        1      21.750000
Name: Age, dtype: float64

In [38]:
X.groupby(['Pclass', 'Sex'])['Age'].transform('mean')

0      26.507589
1      34.611765
2      21.750000
3      34.611765
4      26.507589
         ...    
886    30.740707
887    34.611765
888    21.750000
889    41.281386
890    26.507589
Name: Age, Length: 891, dtype: float64

In [39]:
X[['Pclass', 'Sex', 'Age']]

,Pclass,Sex,Age
0,3,0,22.0
1,1,1,38.0
2,3,1,26.0
3,1,1,35.0
4,3,0,35.0
...,...,...,...
886,2,0,27.0
887,1,1,19.0
888,3,1,NaN
889,1,0,26.0


In [41]:
X['Age'].fillna(X.groupby(['Pclass', 'Sex'])['Age'].transform('mean'))

#X['Age'] = X['Age'].fillna(...)

C:\Users\Concordance\AppData\Local\Temp\ipykernel_11892\3216837243.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [53]:
X_copy = X.copy()
X_copy['Age'] = X_copy['Age'].fillna(X.groupby(['Pclass', 'Sex'])['Age'].transform('mean'))
X_copy[['Pclass', 'Sex', 'Age']]

,Pclass,Sex,Age
0,3,0,22.00
1,1,1,38.00
2,3,1,26.00
3,1,1,35.00
4,3,0,35.00
...,...,...,...
886,2,0,27.00
887,1,1,19.00
888,3,1,21.75
889,1,0,26.00


In [54]:
X_copy = X.copy()
rules = X_copy.groupby(['Pclass', 'Sex'])['Age'].mean()
print(rules)

Pclass  Sex
1       0      41.281386
        1      34.611765
2       0      30.740707
        1      28.722973
3       0      26.507589
        1      21.750000
Name: Age, dtype: float64


In [85]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train[['Pclass', 'Sex', 'Age']]

,Pclass,Sex,Age
331,1,0,45.5
733,2,0,23.0
382,3,0,32.0
704,3,0,26.0
813,3,1,6.0
...,...,...,...
106,3,1,21.0
270,1,0,NaN
860,3,0,41.0
435,1,1,14.0


In [86]:
rules = X_train.groupby(['Pclass', 'Sex'])['Age'].mean()
print(rules)

Pclass  Sex
1       0      40.558158
        1      34.865672
2       0      30.861446
        1      28.387931
3       0      26.640388
        1      21.451220
Name: Age, dtype: float64


In [87]:
rules = X_train.groupby(['Pclass', 'Sex'])['Age'].mean()

pd.MultiIndex.from_frame(X_train[['Pclass','Sex']])

MultiIndex([(1, 0),
            (2, 0),
            (3, 0),
            (3, 0),
            (3, 1),
            (1, 0),
            (1, 0),
            (2, 0),
            (3, 0),
            (1, 0),
            ...
            (3, 0),
            (3, 0),
            (2, 0),
            (1, 1),
            (3, 1),
            (3, 1),
            (1, 0),
            (3, 0),
            (1, 1),
            (1, 0)],
           names=['Pclass', 'Sex'], length=712)

In [88]:
rules = rules.reindex(pd.MultiIndex.from_frame(X_train[['Pclass','Sex']]))
print(rules)

Pclass  Sex
1       0      40.558158
2       0      30.861446
3       0      26.640388
        0      26.640388
        1      21.451220
                 ...    
        1      21.451220
1       0      40.558158
3       0      26.640388
1       1      34.865672
        0      40.558158
Name: Age, Length: 712, dtype: float64


In [89]:
print(rules.set_axis(X_train.index))

331    40.558158
733    30.861446
382    26.640388
704    26.640388
813    21.451220
         ...    
106    21.451220
270    40.558158
860    26.640388
435    34.865672
102    40.558158
Name: Age, Length: 712, dtype: float64


In [90]:
X_train['Age'] = X_train['Age'].fillna(rules.set_axis(X_train.index))

In [91]:
X_train[['Pclass', 'Sex', 'Age']]

,Pclass,Sex,Age
331,1,0,45.500000
733,2,0,23.000000
382,3,0,32.000000
704,3,0,26.000000
813,3,1,6.000000
...,...,...,...
106,3,1,21.000000
270,1,0,40.558158
860,3,0,41.000000
435,1,1,14.000000
